In [1]:
!nvidia-smi

Mon Aug 17 16:30:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!nvcc cuda.cu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [9]:
!./a.out


Found 1 CUDA device(s).

--- Device 0: Tesla T4 ---
  Compute Capability:          7.5
  Total Global Memory:         14912 MB
  Streaming Multiprocessors:   40
  Total Constant memory :      64
  Shared Memory per SM :       64
  Registers Per Block :        65536
  Registers Per SM :           65536
  Cores Per SM:                64
  Total Cores:                 2560
  Max Threads Per Block:       1024
  Max threads per SM :         1024
  Shared Memory Per Block :     48 KB
  Warp Size:                   32
  GPU Clock Rate:             1.59
  Memory Bus Width :           256
  Max Block Dimensions (x,y,z) : [1024, 1024, 64]



In [88]:
%%writefile t1_sync.cu
#include <cstdio>
#include <iostream>

__device__ int val = 0;
__global__ void barrier(int *data, int n){

	int tid = blockIdx.x * blockDim.x + threadIdx.x;
  int sum = 0;
  int sum2 = 0;

	if (tid<n){
		data[tid] = tid;
	}

  //__syncthreads();
	//if (threadIdx.x == 0){
   // atomicAdd(&val, 1);

	//}
  //while ( atomicAdd(&val, 0) < gridDim.x){}

  //__syncthreads();

  if (tid == 0){
    for (int i=0; i<n; i++){
      sum+=i;
      sum2 += data[i];
    }

  printf("sum = %d\n", sum);
  printf("sum2 = %d\n", sum2);
  }

}

int main(){
	int n = 1024;
	int *h_data = new int[n];
	int *d_data;

	cudaMalloc(&d_data, n* sizeof(int));

	int threadsPerBlock = 128;
	int block = 8;

	barrier<<<block, threadsPerBlock>>>(d_data, n);
	cudaDeviceSynchronize();

	cudaMemcpy(h_data, d_data, n* sizeof(int), cudaMemcpyDeviceToHost);


	cudaFree(d_data);
	delete[] h_data;
	return 0;
}


Overwriting t1_sync.cu


In [86]:
!nvcc t1_sync.cu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [87]:
!./a.out #first syncthread

sum = 523776
sums = 523776


In [84]:
!./a.out #no syncthreads

sum = 523776
sums = 523776


In [41]:
!./a.out # with syncthreads

data[0] = 0
data[128] = 0
data[256] = 128
data[384] = 256
data[512] = 384
data[640] = 512
data[768] = 640
data[896] = 768


